# Write ATS input files

In [ ]:
# these can be turned on for development work
%load_ext autoreload
%autoreload 2

In [ ]:
# setting up logging first or else it gets preempted by another package
import watershed_workflow.io
watershed_workflow.io.setupLogging(1)

In [ ]:
import os,sys
import logging
import pickle
import numpy as np

import pandas as pd
import geopandas as gpd
import cftime, datetime
pd.options.display.max_columns = None

import watershed_workflow 
import watershed_workflow.utils
import watershed_workflow.sources
import watershed_workflow.plot
import watershed_workflow.mesh
import watershed_workflow.properties.meteorology
import watershed_workflow.properties.land_cover
import watershed_workflow.hydro
import watershed_workflow.io
import watershed_workflow.sources.standard_names as names

import ats_input_spec
import ats_input_spec.public
import ats_input_spec.io

import amanzi_xml.utils.io as aio
import amanzi_xml.utils.search as asearch
import amanzi_xml.utils.errors as aerrors




In [ ]:
# Force Watershed Workflow to pull data from this directory rather than a shared data directory.
# This picks up the Coweeta-specific datasets set up here to avoid large file downloads for 
# demonstration purposes.
#
def splitPathFull(path):
    """
    Splits an absolute path into a list of components such that
    os.path.join(*splitPathFull(path)) == path
    """
    parts = []
    while True:
        head, tail = os.path.split(path)
        if head == path:  # root on Unix or drive letter with backslash on Windows (e.g., C:\)
            parts.insert(0, head)
            break
        elif tail == path:  # just a single file or directory
            parts.insert(0, tail)
            break
        else:
            parts.insert(0, tail)
            path = head
    return parts

cwd = splitPathFull(os.getcwd())
assert cwd[-1] == 'workflow'
cwd = cwd[:-1]

# Note, this directory is where downloaded data will be put as well
data_dir = os.path.join(*(cwd + ['input_data',]))
def toInput(filename):
    return os.path.join(data_dir, filename)

output_filenames = dict()
output_dir = os.path.join(*(cwd + ['output_data',]))
def fromOutput(filename):
    return os.path.join(output_dir, filename)    

def toOutput(role, filename):
    output_filenames[role] = filename
    return fromOutput(filename)

def pathReltoRun(filename):
    """Strips the absolute path and sets it relative to the run directory, for use in input xml"""
    fs = splitPathFull(filename)
    fs = fs[len(cwd):]
    return os.path.join(*(['..', '..',] + fs))

# check output and input dirs exist
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
if not os.path.isdir(output_dir):
    os.makedirs(output_dir, exist_ok=True)
       

In [ ]:
# Set the data directory to the local space to get the locally downloaded files
# REMOVE THIS CELL for general use outside fo Coweeta
watershed_workflow.utils.setDataDirectory(data_dir)


In [ ]:
## Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed. 
name = 'RussianRiver'
hucs = ['18010110'] # a list of HUCs to run

# Geometric parameters
# -- parameters to clean and reduce the river network prior to meshing
prune_by_area = 10               # km^2
simplify = 125                   # length scale to target average edge 

# -- mesh triangle refinement control
refine_d0 = 200
refine_d1 = 600

refine_L0 = 125
refine_L1 = 300

refine_A0 = refine_L0**2 / 2
refine_A1 = refine_L1**2 / 2

# Refine triangles if they get too acute
min_angle = 20 # degrees

# width of reach by stream order (order:width)
river_widths = dict({1:10, 2:10, 3:20, 4:30, 5:30}) 


# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs = watershed_workflow.crs.default_crs


# start and stop time for simulation
# note that this is the overlap of AORC and MODIS
start = cftime.DatetimeGregorian(2007, 8, 1)
end = cftime.DatetimeGregorian(2020, 7, 31)

start_noleap = cftime.DatetimeNoLeap(2007, 8, 1)
end_noleap = cftime.DatetimeNoLeap(2020, 7, 31)
cyclic_nyears = 10


# Global Soil Properties
min_porosity = 0.05 # minimum porosity considered "too small"
max_permeability = 1.e-10 # max value considered "too permeable"
max_vg_alpha = 1.e-3 # max value of van Genuchten's alpha -- our correlation is not valid for some soils

## Read data from previous runs

In [ ]:
# read in filenames
filenames = dict()
for run in range(1, 7):
    if run == 3:
        for subrun in 'abcd':
            local_filenames_name = fromOutput(f'{run:02}{subrun}_output_filenames.txt')
            with open(local_filenames_name, 'rb') as fid:
                local_dict = pickle.load(fid)
                filenames.update(local_dict)
    else:
        local_filenames_name = fromOutput(f'{run:02}_output_filenames.txt')
        with open(local_filenames_name, 'rb') as fid:
            local_dict = pickle.load(fid)
            filenames.update(local_dict)

display(pd.DataFrame({'keys':filenames.keys(), 'filenames':filenames.values()}))



In [ ]:
print(fromOutput(filenames['nlcd_lai_typical']))
mesh_local_filename = pathReltoRun(fromOutput(filenames['mesh']))
print(mesh_local_filename)

## Helper functions for creating input files

In [ ]:
# Note that each of these are defined as functions so we can reuse them for all three input files.

# add the subsurface and surface domains
#
# Note this also adds a "computational domain" region to the region list, and a vis spec 
# for "domain"
def add_domains(main_list, mesh, surface_region='surface', snow=True, canopy=True):
    ats_input_spec.public.add_domain(main_list, 
                                 domain_name='domain', 
                                 dimension=3, 
                                 mesh_type='read mesh file',
                                 mesh_args={'file':mesh})
    if surface_region:
        main_list['mesh']['domain']['build columns from set'] = surface_region    
    
        # Note this also adds a "surface domain" region to the region list and a vis spec for 
        # "surface"
        ats_input_spec.public.add_domain(main_list,
                                domain_name='surface',
                                dimension=2,
                                mesh_type='surface',
                                mesh_args={'surface sideset name':'surface'})
    if snow:
        # Add the snow and canopy domains, which are aliases to the surface
        ats_input_spec.public.add_domain(main_list,
                                domain_name='snow',
                                dimension=2,
                                mesh_type='aliased',
                                mesh_args={'target':'surface'})
    if canopy:
        ats_input_spec.public.add_domain(main_list,
                                domain_name='canopy',
                                dimension=2,
                                mesh_type='aliased',
                                mesh_args={'target':'surface'})

In [ ]:

def round_dataframe(df, significant_digits):
    def round_sig(x, sig=significant_digits):
        if np.isnan(x):
            return x
        return round(x, sig - int(np.floor(np.log10(abs(x)))) - 1) if x != 0 else 0

    return df.map(lambda x: round_sig(x, 6) if isinstance(x, (float, np.floating)) else x)


In [ ]:
def add_land_cover(main_list):
    lc = pd.read_csv(fromOutput('04_land_cover_properties.csv')).set_index('indices')

    # next write a land-cover section for each NLCD type
    for nlcd_index, nlcd_name in zip(lc.index, lc['names']):
        ats_input_spec.public.add_land_cover(main_list, nlcd_name, nlcd_index, mesh_local_filename, **lc.loc[nlcd_index])
   
    # override Mannings n by stream order
    fe = main_list['state']['evaluators']['surface-manning_coefficient']
    n_by_order = watershed_workflow.properties.land_cover.mannings_n_by_stream_order
    for i in range(10):
        if f'stream order {i}' in main_list['regions']:
            sublist = fe['function'].append_empty(f'stream order {i}')
            sublist['region'] = f'stream order {i}'
            sublist['component'] = 'cell'
            entry = sublist['function'].set_type('constant', ats_input_spec.public.known_specs['function-constant-spec'])
            entry['value'] = n_by_order.loc[min(i, 5), "Manning's n [-]"]



In [ ]:
# add soil sets: note we need a way to name the set, so we use, e.g. SSURGO-MUKEY.
def add_soil_properties(main_list):
    # add soil material ID regions, porosity, permeability, and WRMs
    subsurface_props_used = pd.read_csv(fromOutput(filenames['subsurface properties']))

    subsurface_props_used = round_dataframe(subsurface_props_used, 6)    
    subsurface_props_used = subsurface_props_used.set_index('ats_id')
    
    for ats_id in subsurface_props_used.index:
        props = subsurface_props_used.loc[ats_id]
        set_name = props['name']
        
        smoothing_interval = 0.02
        
        ats_input_spec.public.add_soil_type(main_list, set_name, ats_id, mesh_local_filename,
                                            float(props['porosity [-]']),
                                            float(props['permeability [m^2]']), 1.e-7,
                                            float(props['van Genuchten alpha [Pa^-1]']),
                                            float(props['van Genuchten n [-]']),
                                            float(props['residual saturation [-]']),
                                            float(smoothing_interval))    


In [ ]:
def parseLabeledSets(main_list):
    # add labeled sets
    with open(fromOutput(filenames['labeled set metadata']), 'rb') as fid:
        lss = pickle.load(fid)

    print(lss)
    
    for ls in lss:
        ls_name, ls_setid, ls_entity = ls

        if ls_setid < 10 or ls_setid >= 10000:
            # fixes prior bug in this workflow -- outlet regions should be called "DOMAIN surface outlet", not "DOMAIN outlet"
            if ls_name.endswith("outlet") and ls_name != "surface domain outlet" and ls_name != "RR-outlet":
                ls_name = ls_name[0:-len("outlet")]+"surface outlet"
            
            ats_input_spec.public.add_region_labeled_set(main_list, ls_name, ls_setid, mesh_local_filename, ls_entity)     



In [ ]:
with open(fromOutput('02_watersheds.pickle'), 'rb') as fid:
    watersheds = pickle.load(fid)

watersheds.df


In [ ]:
# get an ATS "main" input spec list -- note, this is a dummy and is not used to write any files yet
def get_main(steadystate=False):
    main_list = ats_input_spec.public.get_main()

    # add the mesh and all domains
    if steadystate:
        add_domains(main_list, mesh_local_filename, canopy=False, snow=False)
    else:
        add_domains(main_list, mesh_local_filename)
        
    # add labeled sets
    parseLabeledSets(main_list)
         
    # add land cover for each nlcd
    add_land_cover(main_list)

    # add soil properties
    add_soil_properties(main_list)
        
    # add observations for each subcatchment
    ats_input_spec.public.add_observations_water_balance(main_list, "computational domain", 
                                                        "surface domain", "external sides",
                                                        steadystate=steadystate)

    #with open(fromOutput('02_watersheds.pickle'), 'rb') as fid:
    #    watersheds = pickle.load(fid)    

    #for ws in watersheds.df['ID']:
    #    ats_input_spec.public.add_observations_water_balance(main_list, ws, steadystate=steadystate)

    # add observations for each gage
    gages = gpd.read_parquet(fromOutput('03d_gages_found.parquet'))    
    gage_obs = ats_input_spec.public.add_observation(main_list, "gage discharge", "gage_discharge.csv", time_units='d',
                                          obs_args={'times start period stop' : [0, 1, -1],
                                                    'times start period stop units' : 'd'})
    for gage in gages['name']:
        if gage == 'RR-outlet': continue
        gage = f'USGS-{gage}'
        ats_input_spec.public.add_observable(gage_obs, f'{gage} discharge [mol d^-1]',
                                             'surface-water_flux', gage,
                                             'extensive integral', 'face', time_integrated=True,
                                             obs_args={'direction normalized flux' : True,
                                                       'direction normalized flux relative to region' : f'{gage} cells'})
    
    return main_list

In [ ]:
def populate_basic_properties(xml, main_xml):
    """This function updates an xml object with the above properties for mesh, regions, soil props, and lc props"""
    # find and replace the mesh list
    xml.replace('mesh', asearch.child_by_name(main_xml, 'mesh'))

    # find and replace the regions list
    xml.replace('regions', asearch.child_by_name(main_xml, 'regions'))

    # update the observations list
    obs = next(i for (i,el) in enumerate(xml) if el.get('name') == 'observations')
    xml[obs] = asearch.child_by_name(main_xml, 'observations')

    # update all model parameters lists
    xml_parlist = asearch.find_path(xml, ['state', 'model parameters'], no_skip=True)
    for parlist in asearch.find_path(main_xml, ['state', 'model parameters'], no_skip=True):
        try:
            xml_parlist.replace(parlist.getName(), parlist)
        except aerrors.MissingXMLError:
            xml_parlist.append(parlist)

    # update all evaluator lists
    xml_elist = asearch.find_path(xml, ['state', 'evaluators'], no_skip=True)
    for elist in asearch.find_path(main_xml, ['state', 'evaluators'], no_skip=True):
        try:
            xml_elist.replace(elist.getName(), elist)
        except aerrors.MissingXMLError:
            xml_elist.append(elist)    
    
    # find and replace land cover
    mp_list = asearch.find_path(xml, ['state', 'model parameters'], no_skip=True)
    lc_list = asearch.find_path(main_xml, ['state', 'model parameters', 'land cover types'], no_skip=True)
    
    try:
        mp_list.replace('land cover types', lc_list)
    except aerrors.MissingXMLError:
        mp_list.append(lc_list)


## Write the files

### Steadystate spinup step

In [ ]:
prefix = 'steadystate'
this_dir = f'00_{name}_{prefix}'

# create the main list
main = get_main(steadystate=True)

# set precip to 0.6 * the mean precip value
precip = main['state']['evaluators'].append_empty('surface-precipitation')
with open(fromOutput(filenames['total_avg_precip']), 'rb') as fid:
    precip_mean = pickle.load(fid)

precip.set_type('independent variable constant', ats_input_spec.public.known_specs['evaluator-independent-variable-constant-spec'])
precip['value'] = float(precip_mean * .6)
    
# load the template file
xml = aio.fromFile(toInput(f'{prefix}-template.xml'))
    
# update the template xml with the main xml generated here
main_xml = ats_input_spec.io.to_xml(main)
populate_basic_properties(xml, main_xml)

# create a run directory
os.makedirs(os.path.join('..', this_dir, 'run0'), exist_ok=True)

# write to disk
aio.toFile(xml, os.path.join('..', this_dir, this_dir+'.xml'))

# increment
previous_prefix = prefix
last_dir = this_dir


            

In [ ]:
print(main)


### Cyclic steadystate spinup step

In [ ]:
prefix = 'cyclic_steadystate'
this_dir = os.path.join(f'01_{name}_{prefix}')

# create the main list
main = get_main(steadystate=False)

# load the template file
xml = aio.fromFile(toInput(f'{prefix}-template.xml'))
    
# add time-series meteorologic data
met_filename = pathReltoRun(fromOutput(filenames['meteorology_typical']))
ats_input_spec.public.add_meteorology_box_evaluators(main, met_filename, include_surface_temperature=True)

# add time-series LAI data
lc = pd.read_csv(fromOutput('04_land_cover_properties.csv')).set_index('indices')
lai_filename = pathReltoRun(fromOutput(filenames['nlcd_lai_typical']))
ats_input_spec.public.add_lai_point_evaluators(main, lai_filename, list(lc['names']))

# update the template xml with the main xml generated here
main_xml = ats_input_spec.io.to_xml(main)
populate_basic_properties(xml, main_xml)

# overwrite restart file info
last_run_dir = os.path.join('..', '..', last_dir, 'run0')

for restart_file in asearch.findall_path(xml, ["initial conditions", "restart file",]):
    print(f"setting IC to {os.path.join(last_run_dir, 'checkpoint_final.h5')}")
    restart_file.setValue(os.path.join(last_run_dir, 'checkpoint_final.h5'))

# create a run directory
os.makedirs(os.path.join('..', this_dir, 'run0'), exist_ok=True)

# write to disk
aio.toFile(xml, os.path.join('..', this_dir, this_dir+'.xml'))

# increment
last_dir = this_dir
previous_prefix = prefix



### Transient simulation

In [ ]:
prefix = 'transient'
this_dir = f'02_{name}_{prefix}'

# create the main list
main = get_main(steadystate=False)

# load the template file

xml = aio.fromFile(toInput(f'{prefix}-template.xml'))

# add time-series meteorologic data
met_filename = pathReltoRun(fromOutput(filenames['meteorology_transient']))
ats_input_spec.public.add_meteorology_box_evaluators(main, met_filename, include_surface_temperature=True)

# add time-series LAI data
lc = pd.read_csv(fromOutput('04_land_cover_properties.csv')).set_index('indices')
lai_filename = pathReltoRun(fromOutput(filenames['nlcd_lai_ts']))
ats_input_spec.public.add_lai_point_evaluators(main, lai_filename, list(lc['names']))

# update the template xml with the main xml generated here
main_xml = ats_input_spec.io.to_xml(main)
populate_basic_properties(xml, main_xml)

# manually setting based on remote run
last_run_dir = os.path.join('..', '..', last_dir, 'run6-try2')

# set checkpoint_final for ICs
for restart_file in asearch.findall_path(xml, ["initial conditions", "restart file",]):
    print(f"setting IC to {os.path.join(last_run_dir, 'checkpoint_final.h5')}")
    restart_file.setValue(os.path.join(last_run_dir, 'checkpoint_final.h5'))
    
# create a run directory
os.makedirs(os.path.join('..', this_dir, 'run0'), exist_ok=True)

# write to disk
aio.toFile(xml, os.path.join('..', this_dir, this_dir+'.xml'))
